In [7]:
import pandas as pd
import pyodbc
from pathlib import Path

DATA_DIR = Path(r"D:\Data Analysis\Tamweely Train\Brazilian E-Commerce Dataset")

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=Brazilian E_Commerce;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
cursor.fast_executemany = True

tables = [
    ("customers", "olist_customers_dataset.csv"),
    ("sellers", "olist_sellers_dataset.csv"),
    ("geolocation", "olist_geolocation_dataset.csv"),
    ("product_category_name_translation", "product_category_name_translation.csv"),
    ("products", "olist_products_dataset.csv"),
    ("orders", "olist_orders_dataset.csv"),
    ("order_items", "olist_order_items_dataset.csv"),
    ("order_payments", "olist_order_payments_dataset.csv"),
    ("order_reviews", "olist_order_reviews_dataset.csv"),
]

for table, file in tables:

    print(f"Loading {table}...")

    df = pd.read_csv(DATA_DIR / file)

    df = df.where(pd.notnull(df), None)

    columns = ",".join(df.columns)
    placeholders = ",".join(["?"] * len(df.columns))

    sql = f"INSERT INTO {table} ({columns}) VALUES ({placeholders})"

    data = list(df.itertuples(index=False, name=None))
    cursor.executemany(sql, data)

    conn.commit()

    print(f"{table} Done ({len(df)} rows)")

cursor.close()
conn.close()

print("Finished Successfully")

Loading customers...
customers Done (99441 rows)
Loading sellers...
sellers Done (3095 rows)
Loading geolocation...
geolocation Done (1000163 rows)
Loading product_category_name_translation...
product_category_name_translation Done (71 rows)
Loading products...


DataError: ('22003', '[22003] [Microsoft][ODBC Driver 17 for SQL Server]Numeric value out of range (0) (SQLExecute)')

In [9]:
import pandas as pd
import pyodbc
from pathlib import Path
import numpy as np

DATA_DIR = Path(r"D:\Data Analysis\Tamweely Train\Brazilian E-Commerce Dataset")

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=Brazilian E_Commerce;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()

tables = [
    ("products", "olist_products_dataset.csv"),
    ("orders", "olist_orders_dataset.csv"),
    ("order_items", "olist_order_items_dataset.csv"),
    ("order_payments", "olist_order_payments_dataset.csv"),
    ("order_reviews", "olist_order_reviews_dataset.csv"),
]

# أعمدة الأرقام في جدول products لازم تتظبط صراحة
numeric_columns = {
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ]
}

for table, file in tables:

    print(f"Loading {table}...")

    df = pd.read_csv(DATA_DIR / file)

    # لو الجدول ده عنده أعمدة أرقام معروفة، اجبرها تتحول لرقم فعلي
    if table in numeric_columns:
        for col in numeric_columns[table]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # استبدال أي NaN (بكل أنواعه) بـ None بشكل مضمون
    df = df.astype(object).where(pd.notnull(df), None)

    columns = ",".join(df.columns)
    placeholders = ",".join(["?"] * len(df.columns))
    sql = f"INSERT INTO {table} ({columns}) VALUES ({placeholders})"

    cursor.fast_executemany = False

    data = list(df.itertuples(index=False, name=None))
    cursor.executemany(sql, data)

    conn.commit()

    print(f"{table} Done ({len(df)} rows)")

cursor.close()
conn.close()

print("Finished Successfully")

Loading products...
products Done (32951 rows)
Loading orders...
orders Done (99441 rows)
Loading order_items...
order_items Done (112650 rows)
Loading order_payments...
order_payments Done (103886 rows)
Loading order_reviews...
order_reviews Done (99224 rows)
Finished Successfully
